# Lab 5: Debugging the Compilation Stack

**Course**: *Dataflow Architectures for AI: From Principles to Practice*  
**Week**: 5, Debugging Across the Dataflow Stack  
**Instructor**: Dr. Kaoutar El Maghraoui (IBM Research)  
**Lab Instructor**: Dr. Rashed Z. Bhatti (IBM Research)  
**Runtime**: Google Colab, GPU runtime required (T4)  
**Expected time**: 90–120 minutes

---

## Learning objectives
By the end of this lab you will be able to:
1. Detect and fix graph breaks with torch._dynamo.explain() and TORCH_LOGS.
2. Quantify numerical divergence between eager and compiled execution.
3. Apply bisection across compilation stages to isolate where a numerical change enters.
4. Classify failures into the diagnostic-funnel layers from lecture.

## Prerequisites
Labs 1-4 complete, plus the Week 5 lecture on debugging across the stack, bisection, and the diagnostic funnel.

## How this lab is graded

This lab includes **checkpoint cells for learner self-checks**: assertion cells inside each Part that verify your outputs land in the expected range as you work. They help you catch mistakes; they do **not** assign a grade.

Official course progress and certificate eligibility are tracked through the **Cognitive Class module review questions and the final exam**, not through this notebook. The analysis prompts in Part 5 are practice for the Module 5 review questions.

### Working through this lab
1. Run each cell in order. Make sure every **Checkpoint** cell passes (no `AssertionError`).
2. Fill in every `TODO` and answer every `YOUR ANALYSIS HERE` prompt.
3. When you are done, complete the **Module 5 review questions** on Cognitive Class.


## Environment Preflight

This cell detects your runtime environment and sets the execution mode:
- **Full GPU**: all exercises run on CUDA (recommended: T4 or better in Colab)
- **Reduced CPU**: GPU-dependent cells use CPU fallback with reduced workloads
- **Static read-only**: no execution; pre-computed outputs are displayed

In [ ]:
# === Environment Preflight ===
import sys
import importlib

# Check Python version
assert sys.version_info >= (3, 9), f"Python >= 3.9 required, got {sys.version}"

# Check PyTorch
try:
    import torch
    _torch_version = tuple(int(x) for x in torch.__version__.split('+')[0].split('.')[:2])
    assert _torch_version >= (2, 2), f"PyTorch >= 2.2 required, got {torch.__version__}"
except ImportError:
    raise RuntimeError("PyTorch not installed. Run: pip install torch>=2.2")

# Detect execution mode
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    MODE = "full-gpu"
    _device_name = torch.cuda.get_device_name(0)
else:
    DEVICE = torch.device("cpu")
    MODE = "reduced-cpu"
    _device_name = "CPU only"

print(f"╔══════════════════════════════════════════════════╗")
print(f"║  Environment Preflight                           ║")
print(f"╠══════════════════════════════════════════════════╣")
print(f"║  Python:    {sys.version.split()[0]:<37s}║")
print(f"║  PyTorch:   {torch.__version__:<37s}║")
print(f"║  Device:    {_device_name:<37s}║")
print(f"║  Mode:      {MODE:<37s}║")
print(f"╚══════════════════════════════════════════════════╝")

if MODE == "reduced-cpu":
    print("\n⚠️  No GPU detected. GPU-dependent cells will use CPU fallback.")
    print("   For full experience: Runtime → Change runtime type → T4 GPU")


---
## 0 · Setup

In [ ]:
!pip -q install transformers==4.44.2

import os
import math
import torch
import torch.nn as nn
import torch._dynamo as dynamo
from torch.profiler import profile, ProfilerActivity, record_function
from transformers import AutoModelForCausalLM, AutoTokenizer

torch.manual_seed(42)
DEV = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'torch {torch.__version__} | device: {DEV}')
if DEV == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
# PyTorch version compatibility check
import torch
print(f'PyTorch version: {torch.__version__}')
assert int(torch.__version__.split('.')[0]) >= 2, (
    f'This lab requires PyTorch 2.x or later. Got {torch.__version__}. '
    f'Run: pip install --upgrade torch'
)
print('✓ PyTorch version compatible')


We use a **deliberately buggy** transformer block that contains common anti-patterns
students encounter when trying to compile real models. Your job is to find and fix them.

In [ ]:
class BuggyTransformerBlock(nn.Module):
    """A transformer block with deliberate compilation anti-patterns.
    
    This block has THREE bugs that affect compilation:
    1. A graph break (silent performance killer)
    2. A numerical precision issue
    3. A dynamic shape that prevents static compilation
    """
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ffn_up = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)
        
        # Bug 2 setup: a threshold stored as a Python float that will
        # cause precision-dependent behavior
        self.scale_threshold = 0.5
    
    def forward(self, x):
        B, S, H = x.shape
        
        # --- Attention block ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        
        # BUG 1: .item() call causes a graph break
        # The student must find this and replace it with a pure-tensor operation
        max_attn = attn.max().item()
        if max_attn > self.scale_threshold:
            attn = attn * 0.9  # "attention dampening" - pedagogical anti-pattern
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, S, H)
        out = self.o_proj(out)
        x = x + out
        
        # --- FFN block ---
        h = self.ln2(x)
        
        # BUG 3: Dynamic shape from Python control flow
        # This creates a graph break when sequence length varies
        if h.shape[1] > 64:
            h = self.ffn_up(h[:, :64, :])  # truncate long sequences
            h = torch.nn.functional.pad(h, (0, 0, 0, S - 64))  # pad back
        else:
            h = self.ffn_up(h)
        
        # BUG 2: Manual softmax without numerical stability (no max subtraction)
        # This diverges from torch.softmax which subtracts max for stability
        h_exp = torch.exp(h)  # unstable: can overflow in fp16
        h = h_exp / h_exp.sum(dim=-1, keepdim=True)  # manual "softmax" normalization
        
        h = self.ffn_down(h)
        x = x + h
        return x


# Instantiate
HIDDEN_DIM = 512
NUM_HEADS = 8
FFN_DIM = 2048
BATCH = 2
SEQ = 128

model = BuggyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to(DEV).eval()
dummy = torch.randn(BATCH, SEQ, HIDDEN_DIM, device=DEV)

# Verify it runs in eager mode
with torch.no_grad():
    out_eager = model(dummy)
print(f'Eager output shape: {out_eager.shape}')
print(f'Eager output mean: {out_eager.mean().item():.6f}')

---
## 1 · ★ TODO: Detect Graph Breaks

Use `torch._dynamo.explain()` to detect how many graph breaks exist in the
`BuggyTransformerBlock`. Report the count, locations, and reasons.

Then try compiling with `torch.compile` and measure the performance vs eager
to confirm the graph breaks are hurting performance.

In [ ]:
# TODO 1.1: Use torch._dynamo.explain() to detect graph breaks
#            Call dynamo.explain(model, dummy) and print:
#            - break_count
#            - break_reasons (iterate and print each)
#            - The number of separate graphs produced

# YOUR CODE HERE
raise NotImplementedError

In [ ]:
# TODO 1.2: Compile the model and benchmark eager vs compiled
#            Use torch.utils.benchmark.Timer from Lab 1.
#            If graph breaks are present, compiled may actually be SLOWER
#            due to the overhead of switching between compiled subgraphs.

from torch.utils import benchmark as torch_bench

# Compile the buggy model
dynamo.reset()  # clear any cached compilations
model_compiled = torch.compile(model)

# Warm up compiled model
with torch.no_grad():
    for _ in range(3):
        _ = model_compiled(dummy)

# YOUR CODE HERE: benchmark both and compute speedup (or slowdown)
# eager_ms = ...
# compiled_ms = ...
# speedup = ...
raise NotImplementedError

---
## 2 · ★ TODO: Fix the Graph Breaks

Create a `FixedTransformerBlock` that eliminates all graph breaks while
preserving the intended behavior. The three bugs are:

1. **`.item()` + Python if**: replace with a pure tensor operation (e.g., `torch.where`)
2. **Manual unstable softmax**: replace with `torch.softmax` or add max-subtraction
3. **Shape-dependent branching**: remove the branch or use `torch._dynamo.mark_dynamic`

After fixing, verify with `dynamo.explain()` that break_count == 0.

In [ ]:
class FixedTransformerBlock(nn.Module):
    """The same transformer block with all graph breaks fixed."""
    def __init__(self, hidden_dim=512, num_heads=8, ffn_dim=2048):
        super().__init__()
        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        
        self.q_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.k_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.v_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.o_proj = nn.Linear(hidden_dim, hidden_dim, bias=False)
        self.ln1 = nn.LayerNorm(hidden_dim)
        self.ln2 = nn.LayerNorm(hidden_dim)
        self.ffn_up = nn.Linear(hidden_dim, ffn_dim, bias=False)
        self.ffn_down = nn.Linear(ffn_dim, hidden_dim, bias=False)
        self.scale_threshold = 0.5
    
    def forward(self, x):
        B, S, H = x.shape
        
        # --- Attention block ---
        h = self.ln1(x)
        q = self.q_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        k = self.k_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        v = self.v_proj(h).reshape(B, S, self.num_heads, self.head_dim).transpose(1, 2)
        
        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.head_dim)
        attn = torch.softmax(scores, dim=-1)
        
        # TODO 2.1: Fix Bug 1: replace .item() + Python if with a tensor op
        # Hint: use torch.where(condition, x_if_true, x_if_false)
        # YOUR CODE HERE
        raise NotImplementedError
        
        out = torch.matmul(attn, v)
        out = out.transpose(1, 2).reshape(B, S, H)
        out = self.o_proj(out)
        x = x + out
        
        # --- FFN block ---
        h = self.ln2(x)
        
        # TODO 2.2: Fix Bug 3: remove shape-dependent branching
        # Just apply ffn_up unconditionally (the truncation was not needed)
        # YOUR CODE HERE
        raise NotImplementedError
        
        # TODO 2.3: Fix Bug 2: replace manual unstable exp/sum with torch.softmax
        # YOUR CODE HERE
        raise NotImplementedError
        
        h = self.ffn_down(h)
        x = x + h
        return x


# Instantiate fixed model with same weights
model_fixed = FixedTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to(DEV).eval()
model_fixed.load_state_dict(model.state_dict())

# Verify no graph breaks
dynamo.reset()
# YOUR CODE HERE: call dynamo.explain on model_fixed and assert break_count == 0
raise NotImplementedError

In [ ]:
# TODO 2.4: Benchmark the fixed model (compiled) vs the buggy model (compiled)
#            The fixed model should show a significant speedup from compilation.

dynamo.reset()
model_fixed_compiled = torch.compile(model_fixed)

# Warm up
with torch.no_grad():
    for _ in range(3):
        _ = model_fixed_compiled(dummy)

# YOUR CODE HERE: benchmark and report speedup
raise NotImplementedError

---
## 3 · ★ TODO: Numerical Divergence Analysis

Now compare the **eager** output of the buggy model against the **eager** output of the
fixed model. The fixes you applied changed the numerical behavior (especially Bug 2: the
unstable softmax). Quantify the divergence.

Then compare the fixed model in **eager** vs **compiled** mode to see what level of
divergence is "normal" from compilation alone.

In [ ]:
# TODO 3.1: Compare buggy eager vs fixed eager
#            Compute: max absolute diff, mean absolute diff, max relative diff

with torch.no_grad():
    out_buggy_eager = model(dummy)
    out_fixed_eager = model_fixed(dummy)

# YOUR CODE HERE: compute and print the three metrics
# max_abs_diff = ...
# mean_abs_diff = ...
# max_rel_diff = ...
raise NotImplementedError

In [ ]:
# TODO 3.2: Compare fixed eager vs fixed compiled
#            This measures divergence from compilation alone (reduction reordering)

with torch.no_grad():
    out_fixed_compiled = model_fixed_compiled(dummy)

# YOUR CODE HERE: compute and print the three metrics
# compilation_max_abs = ...
# compilation_mean_abs = ...
# compilation_max_rel = ...
raise NotImplementedError

In [ ]:
# TODO 3.3: Interpret the results
#            Fill in the comparison table:

comparison = {
    "buggy_vs_fixed_max_abs": None,       # REPLACE with your measured value
    "buggy_vs_fixed_mean_abs": None,      # REPLACE
    "compilation_only_max_abs": None,     # REPLACE
    "compilation_only_mean_abs": None,    # REPLACE
    "is_bug2_divergence_larger": None,    # REPLACE: True/False
    "ratio": None,                        # REPLACE: buggy_vs_fixed / compilation_only
}

print("Divergence comparison:")
for k, v in comparison.items():
    print(f"  {k}: {v}")

### Tolerance Selection and False Positives

Choosing `atol` and `rtol` requires understanding:
1. **Dtype precision**: FP32 has ~7 decimal digits; FP16 has ~3.3; BF16 has ~3.3 but with FP32 range
2. **Accumulation order**: reductions (sum, mean, softmax) are non-associative in floating point
3. **Determinism**: some CUDA ops are non-deterministic by default (atomics in backward pass)

| Comparison | Typical `atol` | Typical `rtol` | Rationale |
|---|---|---|---|
| eager vs eager (same seed) | 0 | 0 | Bitwise identical |
| eager vs compiled (FP32) | 1e-5 | 1e-5 | Reordered ops |
| eager vs compiled (FP16) | 1e-2 | 1e-2 | Low precision + reordering |
| across GPU architectures | 1e-3 | 1e-3 | Different rounding hardware |

> ⚠️ **False positive risk:** Setting `atol` too tight flags benign reordering as bugs.
> Setting it too loose hides real numerical issues.

In [ ]:
# TODO: Explore tolerance sensitivity
# 1. Create a random tensor and compute its sum in forward and reverse order
# 2. Compare the results with different atol values (1e-7, 1e-5, 1e-4, 1e-2)
# 3. Which tolerance correctly identifies this as "same computation, different order" vs "bug"?
#
# Hint: torch.allclose(a, b, atol=X, rtol=Y) returns True if |a-b| <= atol + rtol*|b|

torch.manual_seed(42)
x = torch.randn(128, 768, device=DEVICE if 'DEVICE' in dir() else 'cpu')

# YOUR CODE HERE
# sum_fwd = ...
# sum_rev = ...
# diff = ...

# Discussion: Why does changing the sum order change the result?
# What does this mean for comparing eager vs compiled outputs?

---
## 4 · ★ TODO: Bisection Debugging

Now practice the **bisection method** from lecture. We'll introduce a deliberate
correctness bug in a compilation backend and use bisection to find which stage broke.

The setup: we compile the fixed model with different backends and compare outputs
at each stage to narrow down where divergence exceeds the expected BF16 noise level.

In [ ]:
# Bisection setup: compare outputs at each compilation stage

dynamo.reset()

# Stage 0: Eager baseline (ground truth)
with torch.no_grad():
    baseline = model_fixed(dummy)

# Stage 1: Graph capture only (no optimization) using "eager" backend
dynamo.reset()
model_stage1 = torch.compile(model_fixed, backend="eager")
with torch.no_grad():
    out_stage1 = model_stage1(dummy)

# Stage 2: AOT autograd decomposition (no codegen) using "aot_eager" backend
dynamo.reset()
model_stage2 = torch.compile(model_fixed, backend="aot_eager")
with torch.no_grad():
    out_stage2 = model_stage2(dummy)

# Stage 3: Full compilation with Inductor backend
dynamo.reset()
model_stage3 = torch.compile(model_fixed, backend="inductor")
with torch.no_grad():
    out_stage3 = model_stage3(dummy)

print("Bisection results (max abs diff from eager baseline):")
print(f"  Stage 1 (graph capture only):  {(baseline - out_stage1).abs().max().item():.2e}")
print(f"  Stage 2 (AOT decomposition):   {(baseline - out_stage2).abs().max().item():.2e}")
print(f"  Stage 3 (full Inductor):       {(baseline - out_stage3).abs().max().item():.2e}")

In [ ]:
# TODO 4.1: Interpret the bisection results
#            Answer: at which stage does the divergence first exceed FP32 noise (>1e-6)?
#            What does this tell you about where the numerical change occurs?

bisection_result = {
    "first_divergent_stage": None,   # REPLACE: 1, 2, or 3
    "divergence_cause": None,        # REPLACE: one-sentence explanation
    "is_this_a_bug": None,           # REPLACE: True/False (is the divergence a bug or expected?)
}

print("Bisection interpretation:")
for k, v in bisection_result.items():
    print(f"  {k}: {v}")

---
## 5 · ★ TODO: TORCH_COMPILE_DEBUG Inspection

Use `TORCH_COMPILE_DEBUG` to dump the compilation artifacts and inspect what the
compiler actually did to your fixed model.

In [ ]:
# Run compilation with debug output
import shutil
debug_dir = '/tmp/torch_compile_debug'
if os.path.exists(debug_dir):
    shutil.rmtree(debug_dir)

os.environ['TORCH_COMPILE_DEBUG'] = '1'
os.environ['TORCH_COMPILE_DEBUG_DIR'] = debug_dir

dynamo.reset()
model_debug = torch.compile(model_fixed, backend="inductor")
with torch.no_grad():
    _ = model_debug(dummy)

# Clean up env var
del os.environ['TORCH_COMPILE_DEBUG']
del os.environ['TORCH_COMPILE_DEBUG_DIR']

# List what was produced
import glob
artifacts = glob.glob(f'{debug_dir}/**/*', recursive=True)
print(f"Debug artifacts produced: {len(artifacts)} files")
for a in sorted(artifacts)[:20]:
    print(f"  {a.replace(debug_dir, '.')}")

In [ ]:
# TODO 5.1: Read and inspect the captured FX graph
#            Find the fx_graph_readable.py file and display its contents.
#            Count: how many operators are in the graph? How many are matmul?

# YOUR CODE HERE: find and read the fx_graph file
# Hint: look for files matching '*fx_graph*' or '*readable*' in the debug dir
raise NotImplementedError

In [ ]:
# TODO 5.2: Read the generated kernel code (output_code.py)
#            Find the output_code.py file and identify:
#            - How many Triton kernels were generated?
#            - Which operators were fused together?

# YOUR CODE HERE
raise NotImplementedError

---
## 6 · ★ TODO: Written Analysis: Diagnostic Funnel Classification

For each of the three bugs you found and fixed, classify it in the **diagnostic funnel**
from lecture. Write **250–400 words** total covering:

1. For each bug: which **layer** of the funnel does it originate at? Which layer would the
   **symptom** manifest at if you were running on a dataflow accelerator (e.g., Spyre)?

2. Which debugging **mindset** (software, compiler, or hardware) was required to find each?

3. The bisection results: what did they tell you about the relationship between compilation
   stages and numerical precision? Is the divergence you observed a "bug" or expected behavior?

4. Predict: if you ran the buggy model on Spyre instead of GPU, which bug would have the
   most **severe** consequence (crash vs. wrong answer vs. perf degradation)? Why?

> _Your analysis here (250–400 words). Replace this line._

...

---

## Part 6 - Update the Accelerator Landscape Map

Add this week's row to the shared map you have been building since Lab 1.


In [ ]:
# Part 6 -- Landscape Map row for Week 5. Fill the measured values from your run.
landscape_row_w5 = {
    "accelerator":              f"GPU ({torch.cuda.get_device_name(0)})" if torch.cuda.is_available() else "CPU",
    "debuggability":            "eager fallback hides graph breaks; intermediate values inspectable",
    "graph_breaks_found":       None,   # TODO: int(expl.graph_break_count)
    "compilation_noise_maxabs": None,   # TODO: round(compilation_max_abs, 8)
    "first_divergent_stage":    None,   # TODO: bisection_result["first_divergent_stage"]
    "spyre_prediction":         "no eager fallback: a graph break becomes a compile-time rejection",
}
for _k, _v in landscape_row_w5.items():
    print(f"  {_k:26s}: {_v}")


---
## ★ Optional: Dataflow Deep Dive (Spyre hardware)
_Requires Spyre access via an IBM Research partnership. Skip on the core track._

1. Attempt to compile the `BuggyTransformerBlock` on Spyre (`torch_sendnn`).
2. Document which bugs become **compilation errors** (Spyre rejects) vs **runtime errors**.
3. Use `SENDNN_LOG_LEVEL=DEBUG` to capture the Spyre compiler's IR and identify where
   each bug manifests in the Spyre compilation pipeline.
4. Compare: on GPU, Bug 1 is a silent perf regression. On Spyre, does it compile at all?

> **Spyre vocabulary (consistent across all labs):** `import torch_sendnn` registers the `aiu` device; compile with `torch.compile(model, backend="sendnn")`; dump the backend IR with `TORCH_COMPILE_DEBUG=1` (look for `sdsc_*.json`); raise verbosity with `SENDNN_LOG_LEVEL=DEBUG`. Requires Spyre access via an IBM Research partnership.


In [ ]:
# Spyre track only, pseudocode skeleton
# import torch_sendnn
# model_spyre = BuggyTransformerBlock(HIDDEN_DIM, NUM_HEADS, FFN_DIM).to('aiu').eval()
# try:
#     compiled_spyre = torch.compile(model_spyre, backend='sendnn')
#     out = compiled_spyre(dummy.to('aiu'))
# except Exception as e:
#     print(f"Spyre compilation failed: {e}")
#     # Which bug caused this? Document it.

---
## Submission checklist
- [ ] Runtime → Run all; all outputs visible.
- [ ] Section 1: Graph breaks detected and counted.
- [ ] Section 2: All three bugs fixed; `dynamo.explain` shows 0 breaks.
- [ ] Section 3: Numerical divergence quantified for both comparisons.
- [ ] Section 4: Bisection results interpreted.
- [ ] Section 5: TORCH_COMPILE_DEBUG artifacts inspected.
- [ ] Section 6: Written analysis (250–400 words) completed.
**Submission:** Complete the Cognitive Class review questions for this week. No file upload is required.

_End of Lab 5. Next week: Roofline Analysis, quantifying whether your bottleneck is fixable._

---
*Course: Dataflow Architectures for AI, Week 5 Lab, v1.0 · © 2026 Dr. Kaoutar El Maghraoui*